In [14]:
import os
from dotenv import load_dotenv
import openai
from openai import OpenAI

load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")

documents_dir = "documents"

documents = []
metadatas = []
ids = []
chunk_id = 0

for filename in os.listdir(documents_dir):
    if filename.endswith(".txt"):
        with open(os.path.join(documents_dir, filename), 'r') as file:
            chuncks = file.read().replace('\n', '.').split('###')

            for chunk in chuncks:
                if not chunk.isspace() and  chunk != "":
                    documents.append(chunk)
                    metadatas.append({"source" : filename})
                    ids.append(str(chunk_id))
                    chunk_id += 1


In [15]:
from chromadb.utils import embedding_functions

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key = openai_key,
    model_name = "text-embedding-3-small"
)



In [16]:
import chromadb 

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name = "CVs",
    embedding_function = openai_ef
)

In [17]:
collection.add(
    documents = documents,
    metadatas = metadatas,
    ids = ids
)

In [18]:
user_question = "Mi serve un full stack developer per un sito web"

results = collection.query(
    query_texts = [user_question],
    n_results = 2
)

results



{'ids': [['8', '9']],
 'embeddings': None,
 'documents': [[' Profilo professionale.Sviluppatore full stack con esperienza nella realizzazione di applicazioni web..Competenze sia nello sviluppo frontend sia nella progettazione backend..Interesse per le architetture software e le tecnologie cloud...',
   ' Esperienza lavorativa.Full Stack Developer presso DigitalWorks.Febbraio 2022 - Presente. Sviluppo di applicazioni web utilizzando React e Node.js..Progettazione e implementazione di API REST..Gestione dei database relazionali e integrazione con servizi esterni..Collaborazione con il team tramite metodologie Agile e Scrum...']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'CVLuca.txt'}, {'source': 'CVLuca.txt'}]],
 'distances': [[0.30465590953826904, 0.40059369802474976]]}

In [19]:
context = f"Contesto: nome file {results['metadatas'] [0] [0] ['source']} ecco il paragrafo piu significativo: {results['documents'] [0] [0]}"

context

'Contesto: nome file CVLuca.txt ecco il paragrafo piu significativo:  Profilo professionale.Sviluppatore full stack con esperienza nella realizzazione di applicazioni web..Competenze sia nello sviluppo frontend sia nella progettazione backend..Interesse per le architetture software e le tecnologie cloud...'

In [20]:
prompt = f"""Dato il seguente contesto {context} rispondi alla domanda dell'utente {user_question}
spiegando che nel file individuato c'è il profilo piu adatto. Argomenta la scelta utilizzando
il contenuto del testo individuato nel contesto."""

In [23]:
client = OpenAI(api_key = openai_key)

response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {
        "role" : "developer",
        "content" : "Sei un assistente HR specializzato nella ricerca di professioni"
      },
      {
          "role" : "user",
          "content" : prompt
     }
  ]
)

print(response.choices[0].message.content)

In base alla tua richiesta di un full stack developer per un sito web, ho esaminato il file "CVLuca.txt" e posso confermare che il profilo di Luca è il più adatto per le tue esigenze.

Luca è un sviluppatore full stack con esperienza nella realizzazione di applicazioni web, il che significa che ha competenze sia nel frontend – dove si occupa della parte visiva e interattiva dell’applicazione, garantendo un'esperienza utente fluida – sia nel backend, dove si occupa della logica dell'applicazione, della gestione dei server e del database. Questa combinazione di competenze è cruciale per lo sviluppo di un sito web efficace e performante.

Inoltre, Luca ha un interesse specifico per le architetture software e le tecnologie cloud, il che suggerisce una predisposizione all'uso delle più moderne pratiche di sviluppo e alla scalabilità delle applicazioni. L'adozione di tecnologie cloud è particolarmente importante al giorno d'oggi, in quanto consente maggiore flessibilità, accessibilità e perf